# K06_04 – XGBoost auf Iris als Praxisausblick

Update vom 24. Mai 2026

## Lernziele
- Entscheidungsbaum, Random Forest und XGBoost vergleichen
- verstehen, warum ein einzelner Testsplit kein verlässlicher Benchmark ist
- Cross-Validation als robustere Bewertungsbasis einordnen
- Leistung und Interpretierbarkeit gemeinsam diskutieren

**Wichtig:**  
Dieses Notebook ist als **Praxisausblick** gedacht.  
Auf dem kleinen Iris-Datensatz sind Unterschiede zwischen den Verfahren oft relativ klein.

## Installation und Voraussetzungen

XGBoost ist **nicht Teil von scikit-learn** und muss separat installiert werden.

In Google Colab ist XGBoost meist vorinstalliert. Falls nicht:

```python
# In einer eigenen Code-Zelle ausfuehren:
!pip install xgboost
```

Das Notebook prueft den Import automatisch und laeuft auch ohne XGBoost weiter
(dann nur Entscheidungsbaum und Random Forest im Vergleich).


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False
    print("Hinweis: xgboost ist in dieser Umgebung nicht installiert.")

## 1. Iris-Daten laden

In [2]:
iris = load_iris()
X = iris.data
y = iris.target
target_names = iris.target_names

## 2. Trainings- und Testdaten

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

## 3. Modelle definieren

### XGBoost-Schluesselsparameter

XGBoost hat viele Parameter. Hier die wichtigsten fuer dieses Notebook:

| Parameter | Wert | Bedeutung |
|---|---|---|
| `n_estimators` | 60 | Anzahl der Boosting-Runden (Baeume) |
| `max_depth` | 3 | Maximale Tiefe jedes einzelnen Baums |
| `learning_rate` | 0.1 | Wie stark jeder neue Baum das Ergebnis korrigiert |
| `subsample` | 1.0 | Anteil der Trainingsdaten pro Runde (1.0 = alle) |
| `colsample_bytree` | 1.0 | Anteil der Features pro Baum (1.0 = alle) |
| `objective` | multi:softprob | Mehrklassen-Klassifikation mit Wahrscheinlichkeiten |

> **Merksatz:** XGBoost baut Baeume **sequenziell** -- jeder neue Baum
> korrigiert die Fehler des vorherigen. Das ist der Kernunterschied zum
> Random Forest, der Baeume **parallel** und unabhaengig baut.


In [4]:
models = [
    ("Entscheidungsbaum", DecisionTreeClassifier(random_state=42)),
    ("Random Forest", RandomForestClassifier(n_estimators=150, random_state=42))
]

if XGBOOST_AVAILABLE:
    models.append((
        "XGBoost",
        XGBClassifier(
            n_estimators=60,
            max_depth=3,
            learning_rate=0.1,
            subsample=1.0,
            colsample_bytree=1.0,
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=42,
            n_jobs=1,
            verbosity=0
        )
    ))

## 4. Einzelner Train/Test-Split: erster Vergleich

In [ ]:
split_results = []
for name, clf in models:
    clf.fit(X_train, y_train)
    split_results.append({
        "Verfahren": name,
        "Train-Accuracy": clf.score(X_train, y_train),
        "Test-Accuracy": clf.score(X_test, y_test)
    })

split_df = pd.DataFrame(split_results).round(4)
split_df

,Verfahren,Train-Accuracy,Test-Accuracy
0,Entscheidungsbaum,1.0000,0.9333
1,Random Forest,1.0000,0.8889
2,XGBoost,0.9905,0.9333


In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

cv_results = []
for name, clf in models:
    scores = cross_validate(
        clf, X, y,
        cv=cv,
        scoring='accuracy',
        return_train_score=True
    )
    cv_results.append({
        'Verfahren': name,
        'CV-Accuracy Mittelwert': scores['test_score'].mean(),
        'CV-Accuracy Std': scores['test_score'].std(),
        'Train-Accuracy Mittelwert': scores['train_score'].mean()
    })

# Tabelle sortiert nach CV-Accuracy, bestes Modell zuerst
cv_df = (pd.DataFrame(cv_results)
           .round(4)
           .sort_values('CV-Accuracy Mittelwert', ascending=False)
           .reset_index(drop=True))
cv_df


,Verfahren,CV-Accuracy Mittelwert,CV-Accuracy Std,Train-Accuracy Mittelwert
0,Random Forest,0.9533,0.0521,1.0000
1,XGBoost,0.9533,0.0521,0.9993
2,Entscheidungsbaum,0.9333,0.0516,1.0000


## 6. Interpretation der Cross-Validation

Lesen Sie die Cross-Validation bitte so:

- **CV-Accuracy Mittelwert**: Wie gut ist das Verfahren im Mittel über mehrere Aufteilungen?
- **CV-Accuracy Std**: Wie stark schwanken die Ergebnisse?
- **Train-Accuracy Mittelwert**: Gibt es Hinweise auf Overfitting?

**Wichtiger Punkt für die Vorlesung:**  
Wenn der Entscheidungsbaum in einem einzelnen Split einmal besser aussieht, heißt das nicht automatisch, dass er das bessere Verfahren ist.  
Für die Vorlesung ist die **Cross-Validation-Tabelle** die verlässlichere Grundlage.

## 7. Interpretation: Leistung und Interpretierbarkeit

Nicht nur die Accuracy ist wichtig.

Auch folgende Fragen zählen:
- Wie gut ist das Verfahren **erklärbar**?
- Wie stark schwanken die Ergebnisse?
- Wie aufwendig ist das Training?
- Wie wichtig ist maximale Leistung im Vergleich zur Nachvollziehbarkeit?

Typische Einordnung:
- **Entscheidungsbaum**: gut interpretierbar
- **Random Forest**: robuster, aber weniger transparent
- **XGBoost**: oft leistungsstark, aber deutlich schwerer zu interpretieren

## 8. Mini-Übungen

1. Warum reicht ein einzelner Testsplit nicht aus, um drei Verfahren fair zu vergleichen?
2. Warum sollte man die Ergebnisse auf Iris nicht überinterpretieren?
3. Welches Verfahren wirkt in der Cross-Validation am stabilsten?

## 9. Fazit

- Auf kleinen Datensätzen können Einzelsplits täuschen.
- Ein einzelner Entscheidungsbaum kann zufällig sehr gut abschneiden.
- Für einen belastbaren Vergleich ist die **Cross-Validation** besser geeignet.
- Modellwahl ist immer auch ein Abwägen von **Leistung**, **Stabilität** und **Interpretierbarkeit**.

## 10. Musterlösungen und didaktische Hinweise

**Zu 1:** Weil das Ergebnis stark von der konkreten Datenaufteilung abhängen kann.

**Zu 2:** Weil Iris klein und relativ leicht trennbar ist; Unterschiede zwischen den Verfahren bleiben deshalb oft gering.

**Zu 3:** Das zeigt typischerweise die kleinste Standardabweichung bei gleichzeitig gutem Mittelwert.

**Didaktischer Hinweis:**  
K06_04 sollte in der Vorlesung als **Praxisausblick** gelesen werden, nicht als überzogener Wettkampf zwischen drei Verfahren.

## Abschluss: Welches Modell fuer den Einsatz?

Der CV-Vergleich zeigt, welches Verfahren auf Iris am robustesten ist.
Auf einem kleinen Datensatz wie Iris sind die Unterschiede oft gering --
das ist eine wichtige Erkenntnis fuer sich.

Fuer den produktiven Einsatz gilt:

| Kriterium | Entscheidungsbaum | Random Forest | XGBoost |
|---|:---:|:---:|:---:|
| Interpretierbarkeit | ++ | - | -- |
| Robustheit / Generalisierung | - | ++ | ++ |
| Trainingsaufwand | gering | mittel | hoeher |
| Hyperparameter-Tuning | wenig | maessig | viel |

> **Empfehlung fuer Iris:** Alle drei Verfahren funktionieren gut.
> In der Praxis waehlt man Random Forest oder XGBoost, wenn
> Robustheit wichtiger ist als Erklaerbarkeit.


In [ ]:
# Bestes Verfahren aus CV-Tabelle ablesen
bestes_verfahren = cv_df.iloc[0]['Verfahren']
print(f'Bestes Verfahren laut CV: {bestes_verfahren}')
print(f'CV-Accuracy: {cv_df.iloc[0]["CV-Accuracy Mittelwert"]:.4f} '
      f'(+/- {cv_df.iloc[0]["CV-Accuracy Std"]:.4f})')
print()

# Finales Modell mit dem besten Verfahren auf allen Daten trainieren
# (hier als Beispiel: Random Forest, unabhaengig vom CV-Ergebnis)
from sklearn.ensemble import RandomForestClassifier
final_model = RandomForestClassifier(n_estimators=150, random_state=42)
final_model.fit(X, y)   # alle 150 Iris-Samples

print('Finales Modell (Random Forest) ist einsatzbereit.')

# Beispielvorhersage
neuer_punkt = [[5.1, 3.5, 1.4, 0.2]]
klasse = final_model.predict(neuer_punkt)[0]
proba  = final_model.predict_proba(neuer_punkt)[0]
print(f'Vorhersage: {target_names[klasse]}')
print(f'Wahrscheinlichkeiten: {dict(zip(target_names, proba.round(3)))}')


Bestes Verfahren laut CV: Random Forest
CV-Accuracy: 0.9533 (+/- 0.0521)

Finales Modell (Random Forest) ist einsatzbereit.
Vorhersage: setosa
Wahrscheinlichkeiten: {np.str_('setosa'): np.float64(1.0), np.str_('versicolor'): np.float64(0.0), np.str_('virginica'): np.float64(0.0)}
